# Toy Masks And Trails

This notebook is a deliberately small first step toward a poi trail extraction pipeline.

It does not use real video yet. Instead it generates synthetic frames with two colored lights and a dim body-like occluder so the early computer vision stages are easy to inspect.

The goals are:

- inspect synthetic frames
- derive color and luminosity masks
- accumulate those masks over time into a simple trail render
- save a few debug artifacts under `experiments/cv/artifacts/generated/toy`

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

ARTIFACT_DIR = Path.cwd().resolve().parent / 'artifacts' / 'generated' / 'toy'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

HEIGHT = 360
WIDTH = 640
FRAME_COUNT = 96
BODY_TOP_LEFT = (260, 90)
BODY_BOTTOM_RIGHT = (380, 320)
print(f'Artifacts will be written to: {ARTIFACT_DIR}')

In [ ]:
def add_glow(frame: np.ndarray, center: tuple[int, int], color: tuple[int, int, int], radius: int = 12) -> None:
    glow = np.zeros_like(frame)
    cv2.circle(glow, center, radius * 2, color, thickness=-1, lineType=cv2.LINE_AA)
    glow = cv2.GaussianBlur(glow, (0, 0), sigmaX=10, sigmaY=10)
    np.maximum(frame, glow, out=frame)
    cv2.circle(frame, center, radius, color, thickness=-1, lineType=cv2.LINE_AA)

def generate_frame(frame_index: int) -> tuple[np.ndarray, dict[str, tuple[int, int]]]:
    image = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
    image[:] = (6, 6, 10)

    t = 2 * np.pi * frame_index / FRAME_COUNT
    left = (
        int(WIDTH * 0.32 + 70 * np.cos(2.0 * t)),
        int(HEIGHT * 0.50 + 92 * np.sin(1.0 * t)),
    )
    right = (
        int(WIDTH * 0.68 + 78 * np.cos(2.0 * t + np.pi / 3)),
        int(HEIGHT * 0.48 + 88 * np.sin(1.0 * t + np.pi / 2)),
    )

    add_glow(image, left, (40, 40, 255))
    add_glow(image, right, (255, 255, 0))

    cv2.rectangle(image, BODY_TOP_LEFT, BODY_BOTTOM_RIGHT, (28, 28, 32), thickness=-1)
    cv2.rectangle(image, BODY_TOP_LEFT, BODY_BOTTOM_RIGHT, (52, 52, 60), thickness=2)

    return image, {'left': left, 'right': right}

frames = []
positions = []
for frame_index in range(FRAME_COUNT):
    frame, frame_positions = generate_frame(frame_index)
    frames.append(frame)
    positions.append(frame_positions)

print(f'Generated {len(frames)} synthetic frames.')

In [ ]:
sample_indices = [0, FRAME_COUNT // 4, FRAME_COUNT // 2, 3 * FRAME_COUNT // 4]
fig, axes = plt.subplots(1, len(sample_indices), figsize=(16, 4))
for axis, sample_index in zip(axes, sample_indices):
    axis.imshow(cv2.cvtColor(frames[sample_index], cv2.COLOR_BGR2RGB))
    axis.set_title(f'Frame {sample_index}')
    axis.axis('off')
plt.tight_layout()
preview_path = ARTIFACT_DIR / 'toy_frames_preview.png'
fig.savefig(preview_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved preview to {preview_path}')

In [ ]:
def build_masks(frame_bgr: np.ndarray) -> dict[str, np.ndarray]:
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
    luma = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)

    red_mask_a = cv2.inRange(hsv, (0, 120, 140), (12, 255, 255))
    red_mask_b = cv2.inRange(hsv, (170, 120, 140), (179, 255, 255))
    red_mask = cv2.bitwise_or(red_mask_a, red_mask_b)

    cyan_mask = cv2.inRange(hsv, (80, 100, 140), (105, 255, 255))
    bright_mask = cv2.inRange(luma, 120, 255)

    kernel = np.ones((3, 3), dtype=np.uint8)
    red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_OPEN, kernel)
    cyan_mask = cv2.morphologyEx(cyan_mask, cv2.MORPH_OPEN, kernel)
    bright_mask = cv2.morphologyEx(bright_mask, cv2.MORPH_OPEN, kernel)

    return {
        'hsv': hsv,
        'luma': luma,
        'red_mask': red_mask,
        'cyan_mask': cyan_mask,
        'bright_mask': bright_mask,
    }

sample_frame = frames[10]
mask_bundle = build_masks(sample_frame)

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes[0, 0].imshow(cv2.cvtColor(sample_frame, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('RGB frame')
axes[0, 1].imshow(mask_bundle['hsv'])
axes[0, 1].set_title('HSV frame')
axes[0, 2].imshow(mask_bundle['luma'], cmap='gray')
axes[0, 2].set_title('Luma')
axes[1, 0].imshow(mask_bundle['red_mask'], cmap='Reds')
axes[1, 0].set_title('Red mask')
axes[1, 1].imshow(mask_bundle['cyan_mask'], cmap='Blues')
axes[1, 1].set_title('Cyan mask')
axes[1, 2].imshow(mask_bundle['bright_mask'], cmap='gray')
axes[1, 2].set_title('Brightness mask')
for axis in axes.ravel():
    axis.axis('off')
plt.tight_layout()
mask_path = ARTIFACT_DIR / 'toy_mask_breakdown.png'
fig.savefig(mask_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved mask breakdown to {mask_path}')

In [ ]:
red_accumulator = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
cyan_accumulator = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
fade = 0.93

for frame in frames:
    masks = build_masks(frame)
    red_accumulator = np.maximum(red_accumulator * fade, masks['red_mask'] / 255.0)
    cyan_accumulator = np.maximum(cyan_accumulator * fade, masks['cyan_mask'] / 255.0)

trail_rgb = np.zeros((HEIGHT, WIDTH, 3), dtype=np.float32)
trail_rgb[..., 0] = cyan_accumulator
trail_rgb[..., 1] = cyan_accumulator
trail_rgb[..., 2] = red_accumulator
trail_rgb = np.clip(trail_rgb, 0.0, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(frames[-1], cv2.COLOR_BGR2RGB))
axes[0].set_title('Last synthetic frame')
axes[1].imshow(trail_rgb)
axes[1].set_title('Accumulated toy trails')
for axis in axes:
    axis.axis('off')
plt.tight_layout()
trail_path = ARTIFACT_DIR / 'toy_accumulated_trails.png'
fig.savefig(trail_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved accumulated trails to {trail_path}')

## What to do next

Once this toy example makes sense, the next notebook should swap synthetic frames for a real clip and keep the same inspection flow:

1. open a video file
2. inspect channels and masks on selected frames
3. evaluate how often the poi disappear behind the body
4. only then move to connected components, centroids, and tracking